# SEAL math evaluation

Compare the unsteered model and SEAL on MATH500 and GSM8K using the same engine, prompts and sampling settings. The model is DeepSeek-R1-Distill-Qwen-1.5B. Steering applies `execution - reflection - transition` at layer 20, scale 0.5, on paragraph-break tokens during generation.

Run from this directory after preparing the data in `README.md`. Full answers are saved under `.runtime/`; the notebook prints computed metrics only. The vectors are the existing experiment artifacts; this evaluation does not retrain them.

Updated API cells have not been rerun at full scale. Historical full-dataset metrics remain in `README.md`.


In [ ]:
import json
import os
from pathlib import Path

import torch
import vllm
from common import load_evaluation_data, make_prompts, score_outputs
from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec

from easysteer.extraction import StatisticalControlVector

MODEL = os.environ.get("EASYSTEER_MODEL", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
DATA_DIR = Path(os.environ.get("EASYSTEER_DATA_DIR", "."))
RESULTS_DIR = Path(".runtime")
LIMIT = int(os.environ.get("EASYSTEER_LIMIT", "0"))  # Zero evaluates each full dataset.
MAX_TOKENS = 8192
RESULTS_DIR.mkdir(exist_ok=True)

llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steer_algorithms=["direct"],
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
)
tokenizer = llm.get_tokenizer()
sampling = SamplingParams(
    temperature=0, max_tokens=MAX_TOKENS, skip_special_tokens=False
)
print(
    f"vLLM {vllm.__version__}; Torch {torch.__version__}; {torch.cuda.get_device_name(0)}"
)
print(f"Sample limit: {LIMIT or 'full datasets'}; max generated tokens: {MAX_TOKENS}")

In [ ]:
newline_token_ids = sorted(
    token_id
    for token, token_id in tokenizer.get_vocab().items()
    if token.endswith("ĊĊ")
)
parts = {
    name: StatisticalControlVector.import_gguf(f"{name}_avg_vector.gguf")
    for name in ("execution", "reflection", "transition")
}
# Additive directions share a target and selector, so one payload represents their sum.
merged = StatisticalControlVector(
    method="Average",
    directions={
        20: (
            parts["execution"].directions[20]
            - parts["reflection"].directions[20]
            - parts["transition"].directions[20]
        )
    },
    metadata={"composition": "execution - reflection - transition"},
    model_type=MODEL,
    component="hidden_states",
)
steering = merged.to_spec(
    scale=0.5,
    apply=ApplySpec(generation_tokens=newline_token_ids),
)


## Accuracy and generation length

The paper's explicit reasoning prefix is retained. GSM8K reference answers are the numeric answers following `####`, rather than the full worked solutions. Generated answers are evaluated with `math_verify`.


In [ ]:
results = []
for dataset in ("math500", "gsm8k_test"):
    problems, answers = load_evaluation_data(DATA_DIR / f"{dataset}.json", limit=LIMIT)
    prompts = make_prompts(tokenizer, problems)
    for condition, spec in (("baseline", False), ("steered", steering)):
        outputs = llm.generate(prompts, sampling, steering=spec, use_tqdm=False)
        metrics, rows = score_outputs(problems, answers, outputs)
        results.append({"dataset": dataset, "condition": condition, **metrics})
        (RESULTS_DIR / f"{dataset}_{condition}.json").write_text(
            json.dumps({"metrics": metrics, "rows": rows}, ensure_ascii=False, indent=2)
            + "\n"
        )
        print(
            f"{dataset:12s} {condition:8s}: accuracy={metrics['accuracy']:.2%}, "
            f"mean tokens={metrics['mean_generated_tokens']:.1f}, n={metrics['questions']}"
        )
        del outputs

print("\nSummary")
for row in results:
    print(
        f"{row['dataset']:12s} {row['condition']:8s}  "
        f"{row['accuracy']:7.2%}  {row['mean_generated_tokens']:8.1f} tokens"
    )
